# OMIE day-ahead prices: file exploration
Explore the format of a single daily OMIE price file (`marginalpdbc`, 2024-01-01) before automating the download of the full period.

In [ ]:
import pandas as pd
pd.__version__

In [ ]:
df = pd.read_csv("../data/raw/marginalpdbc_20240101.1")
print(df.shape)
print(df.head())
print(df.tail())

## Reading the file
The file has no column names, a one-line header and a one-line footer, so it needs these parameters:
- `sep=";"`: columns are separated by semicolons.
- `skiprows=1`: skips the first line, which only contains the file type (`MARGINALPDBC`).
- `skipfooter=1`: skips the last line, which only contains an `*`.
- `engine="python"`: required because the default C engine does not support `skipfooter`.
- `header=None`: the file has no header row, so pandas must not use the first data row as column names.
- `names=[...]`: assigns clear column names in English and snake_case.

In [ ]:
df= pd.read_csv(
"../data/raw/marginalpdbc_20240101.1",
names=["year", "month" ,"day", "period", "price_pt", "price_es", "empty"],
sep= ";",
skiprows=1,
skipfooter=1,
engine="python",
header=None)
print (df)

## Empty column
Each line ends with a `;`, so pandas creates an extra empty column. I check that it is completely empty (to make sure no data is lost) before dropping it, since it carries no information.

In [ ]:
df["empty"].isna()

In [ ]:
df["empty"].isna().all()

In [ ]:
df=df.drop(columns=["empty"])
df.head()

## Data quality checks
I check that each column has the expected type (`int64` for dates and periods, `float64` for prices). A non-numeric value, such as a comma used as decimal separator, would silently turn the whole price column into text. Result: types are as expected and 0 rows have non-numeric prices.

In [ ]:
df.dtypes

In [ ]:
df.dtypes

In [ ]:
mask = pd.to_numeric(df["price_es"], errors="coerce").isna()
df[mask]


## Findings
- The file has 24 rows, one per hourly period. This changes later: from October 2025 the market uses 15-minute periods (96 per day), and daylight saving days have 23 or 25 hourly periods. The download pipeline must validate the expected row count per date.
- Data types are correct, there are no non-numeric prices, and the empty trailing column was removed.
- **Market splitting** in periods 11 and 12: Spain and Portugal have different prices, which happens when the interconnection between both countries is congested.
- Prices drop sharply around midday on this public holiday. Two possible explanations: higher solar output displacing gas-fired generation, and lower demand because factories, offices and shops are closed. Demand is a **confounding variable**: without demand data (ESIOS) the two effects cannot be separated.
- This is **anecdotal evidence** from a single day; no conclusions can be drawn until the full 2022-2025 period is analysed.